In [1]:
import optuna
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

from model import WDMPNN, NodeEdgeSSLModel
from data_preparation import load_and_split_data, PolymerDataset, NodeEdgeMaskDataset


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1) 加载并划分原始数据（只用 SMILES，不需要 y）
base_path = "neurips-open-polymer-prediction-2025"
train_df, _, _ = load_and_split_data(base_path)
base_graph_ds = PolymerDataset(train_df, y_cols=[])                # y_cols=[] 表示不生成 y

# 2) 构造 Node/Edge SSL 数据集和 DataLoader
dataset_nodeedge = NodeEdgeMaskDataset(base_graph_ds, device=device)
nodeedge_loader   = DataLoader(dataset_nodeedge, batch_size=64, shuffle=True)

👉 加载主训练数据
  原始训练样本数: 7973
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条
  → 正在增强 Density 数据，共 787 条
cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条
  → 正在增强 FFV 数据，共 862 条


[13:44:21] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[13:44:21] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[13:44:21] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[13:44:21] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[13:44:21] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[13:44:21] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[13:44:21] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[13:44:21] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[13:44:21] SMILES Parse 

cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081
👉 划分 train / validation / test
  划分结果: train=8064, val=1008, test=1009
📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条


In [3]:
def objective_stage1(trial):
    # 超参空间
    lr         = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])
    n_layers   = trial.suggest_int("num_edge_layers", 2, 4)

    # 早停参数
    patience = 15  # 允许连续 15 轮验证损失不下降
    min_delta = 1e-3  # 变化阈值
    best_loss = float('inf')
    no_improve = 0
    
    # 模型 & SSL head
    encoder = WDMPNN(
        node_feat_dim=2,        # 与 create_graph_from_smiles 中的 node_feat_dim 对应
        edge_feat_dim=1,        # 与 edge_feat_dim 对应
        hidden_dim=hidden_dim,
        num_edge_layers=n_layers
    ).to(device)
    model = NodeEdgeSSLModel(encoder, node_feat_dim=2, edge_feat_dim=1).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # 训练若干 epoch
    epoch = 0
    while True:
        epoch += 1
        total_loss = 0
        for batch in nodeedge_loader:
            batch = batch.to(device)
            node_pred, edge_pred = model(
                batch.x_masked,
                batch.edge_index,
                batch.edge_attr_masked,
                batch.batch
            )
            loss = (
                F.mse_loss(node_pred, batch.x_orig) +
                F.mse_loss(edge_pred, batch.edge_attr_orig)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs

        avg_loss = total_loss / len(nodeedge_loader.dataset)
        trial.report(avg_loss, epoch)

        # 早停判断
        if avg_loss < best_loss - min_delta:
            best_loss = avg_loss
            no_improve = 0
            # 保存最佳模型
            torch.save(encoder.state_dict(), f"stage1_encoder_trial{trial.number}.pt")
        else:
            no_improve += 1
            if no_improve >= patience:
                break  # 早停

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # 保存最优 encoder 权重
    torch.save(encoder.state_dict(), f"stage1_encoder_trial{trial.number}.pt")
    return best_loss

In [4]:

# 4) 启动 Optuna
storage_uri = "sqlite:///stage1_optuna.db"
study1 = optuna.create_study(
    study_name="stage1_nodeedge_ssl",
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(),
    storage=storage_uri,
    load_if_exists=True,
)
study1.optimize(objective_stage1, show_progress_bar=True, n_trials=20)

print("Stage1 best params:", study1.best_trial.params)

[I 2025-07-31 13:44:28,823] Using an existing study with name 'stage1_nodeedge_ssl' instead of creating a new one.


  0%|          | 0/20 [00:00<?, ?it/s]

[W 2025-07-31 13:46:13,360] Trial 5 failed with parameters: {'lr': 6.130271314868202e-05, 'hidden_dim': 256, 'num_edge_layers': 4} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/root/miniconda3/envs/test1/lib/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1951311/1682701371.py", line 29, in objective_stage1
    for batch in nodeedge_loader:
  File "/root/miniconda3/envs/test1/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 733, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/envs/test1/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 789, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/envs/test1/lib/python3.11/site-packages/to

KeyboardInterrupt: 